#                        Sheth L.U.J And Sir M.V college
Kunal Joshi | T086
Practical No. 3B

Aim: Feature Scaling and Dummification


*   Apply feature-scaling techniques like standardization and normalization to numerical features
*  Perform feature dummification to convert categorical variables into numerical representations.







#Handling Categorical Data & Imbalanced Classes

In [19]:
import pandas as pd
import numpy as np

# creating a representative sample of the dataset provided
data = pd.read_csv("laptop_price.csv", encoding='latin1')
df = pd.DataFrame(data)

# Pre-processing: Clean 'Ram' to be numerical for later use in KNN/Binning
df['Ram_Num'] = df['Ram'].str.replace('GB', '').astype(int)

print("Original Data Preview:")
print(df[['Company', 'TypeName', 'Ram', 'Price_euros']].head())

Original Data Preview:
  Company   TypeName   Ram  Price_euros
0   Apple  Ultrabook   8GB      1339.69
1   Apple  Ultrabook   8GB       898.94
2      HP   Notebook   8GB       575.00
3   Apple  Ultrabook  16GB      2537.45
4   Apple  Ultrabook   8GB      1803.60


#Encoding Nominal Categorical Features

In [20]:
from sklearn.preprocessing import LabelBinarizer

print("\n--- 8. Encoding Nominal Features (One-Hot) ---")

# Target: 'TypeName' (Ultrabook, Notebook, etc.)
feature = np.array(df['TypeName'])

# Create the One-Hot Encoder
one_hot = LabelBinarizer()
encoded_data = one_hot.fit_transform(feature)

print("Classes found:", one_hot.classes_)
print("Encoded Matrix (First 5 rows):\n", encoded_data[:5])

# To see it in a DataFrame format:
# pd.get_dummies(df['TypeName'])


--- 8. Encoding Nominal Features (One-Hot) ---
Classes found: ['2 in 1 Convertible' 'Gaming' 'Netbook' 'Notebook' 'Ultrabook'
 'Workstation']
Encoded Matrix (First 5 rows):
 [[0 0 0 0 1 0]
 [0 0 0 0 1 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 1 0]]


#Encoding Dictionaries of Features

In [21]:
from sklearn.feature_extraction import DictVectorizer

print("\n--- 9. Encoding Dictionaries of Features ---")

# Convert 'Company' and 'OpSys' into a dictionary format
# Example: [{'Company': 'Apple', 'OpSys': 'macOS'}, ...]
data_dict = df[['Company', 'OpSys']].to_dict(orient='records')

# Create Vectorizer (sparse=False gives a dense numpy array)
dictvectorizer = DictVectorizer(sparse=False)
features_dict = dictvectorizer.fit_transform(data_dict)

print("Feature Names generated:", dictvectorizer.get_feature_names_out())
print("Vectorized Data (First 3 rows):\n", features_dict[:3])


--- 9. Encoding Dictionaries of Features ---
Feature Names generated: ['Company=Acer' 'Company=Apple' 'Company=Asus' 'Company=Chuwi'
 'Company=Dell' 'Company=Fujitsu' 'Company=Google' 'Company=HP'
 'Company=Huawei' 'Company=LG' 'Company=Lenovo' 'Company=MSI'
 'Company=Mediacom' 'Company=Microsoft' 'Company=Razer' 'Company=Samsung'
 'Company=Toshiba' 'Company=Vero' 'Company=Xiaomi' 'OpSys=Android'
 'OpSys=Chrome OS' 'OpSys=Linux' 'OpSys=Mac OS X' 'OpSys=No OS'
 'OpSys=Windows 10' 'OpSys=Windows 10 S' 'OpSys=Windows 7' 'OpSys=macOS']
Vectorized Data (First 3 rows):
 [[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 1.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
  0. 0. 0. 0.]]


#Encoding Ordinal Categorical Features (Binning)

In [22]:
# 1. Binning: Discretize 'Price_euros' into 3 bins
df['Price_Category'] = pd.cut(df['Price_euros'], bins=3, labels=["Low", "Medium", "High"])
print("Binned Categories:\n", df[['Price_euros', 'Price_Category']].head())

# 2. Encoding: Map these ordinal categories to integers
scale_mapper = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

df['Price_Ranked'] = df['Price_Category'].replace(scale_mapper)
print("\nEncoded Bins:\n", df[['Price_Category', 'Price_Ranked']].head())

Binned Categories:
    Price_euros Price_Category
0      1339.69            Low
1       898.94            Low
2       575.00            Low
3      2537.45         Medium
4      1803.60            Low

Encoded Bins:
   Price_Category Price_Ranked
0            Low            1
1            Low            1
2            Low            1
3         Medium            2
4            Low            1


/tmp/ipython-input-1391407499.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Price_Ranked'] = df['Price_Category'].replace(scale_mapper)
/tmp/ipython-input-1391407499.py:12: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['Price_Ranked'] = df['Price_Category'].replace(scale_mapper)


#Imputing Missing Class Values (using KNN)

In [23]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

print("\n--- 11. Imputing Missing Class Values (KNN) ---")

# 1. Prepare Data
# We need to encode the target (OpSys) to numbers temporarily for the Classifier
le = LabelEncoder()
df['OpSys_Encoded'] = le.fit_transform(df['OpSys'])

# Features to use for prediction
X = df[['Price_euros', 'Inches', 'Ram_Num']].values
y = df['OpSys_Encoded'].values # Target

# 2. Create Artificial Missing Value
X_with_nan = X.copy()
y_with_missing = y.astype(float)
y_with_missing[2] = np.nan  # Determine index 2 as missing
y_with_missing[5] = np.nan  # Determine index 5 as missing

print(f"Original Class at index 2: {le.inverse_transform([int(y[2])])[0]}")

# 3. Split into Train (Known) and Test (Unknown)
mask_known = ~np.isnan(y_with_missing)
mask_unknown = np.isnan(y_with_missing)

X_train = X_with_nan[mask_known]
y_train = y_with_missing[mask_known]
X_predict = X_with_nan[mask_unknown]

# 4. Train KNN
knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
knn.fit(X_train, y_train)

# 5. Predict
predicted_encoded = knn.predict(X_predict)
predicted_labels = le.inverse_transform(predicted_encoded.astype(int))

print(f"Predicted Class for missing values: {predicted_labels}")


--- 11. Imputing Missing Class Values (KNN) ---
Original Class at index 2: No OS
Predicted Class for missing values: ['Windows 10' 'Windows 10']


#Handling Imbalanced Classes

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("\n--- 12. Handling Imbalanced Classes ---")

# Prepare features and target
X_features = df[['Price_euros', 'Ram_Num']].values
y_target_raw = df['Company'].values

# Encode target strings to integers
le_company = LabelEncoder()
y_target = le_company.fit_transform(y_target_raw)

weights = 'balanced'

# Initialize Classifier with class_weight
clf = RandomForestClassifier(class_weight=weights, random_state=0)
clf.fit(X_features, y_target)

print("Classes:", le_company.classes_)
print("Model trained using 'balanced' class weights.")



--- 12. Handling Imbalanced Classes ---
Classes: ['Acer' 'Apple' 'Asus' 'Chuwi' 'Dell' 'Fujitsu' 'Google' 'HP' 'Huawei'
 'LG' 'Lenovo' 'MSI' 'Mediacom' 'Microsoft' 'Razer' 'Samsung' 'Toshiba'
 'Vero' 'Xiaomi']
Model trained using 'balanced' class weights.
